# tp-22 track_bias 先読みリーク修正の影響測定（旧表 vs 修正後の表）

**背景**: 2026-10-15 に、馬場傾向「当日の直前傾向」（`track_bias_winner_3f_same_day_prior` など）が、同じ日の**後のレースの結果**を使っていた不具合を直した。これまでの実験（tp-03〜tp-21）は修正前の LayerA・学習表（`layer_a.parquet` は 10-14 作成）で出した値。ここでは、修正後のコードで作り直した表（`training_pipeline_fixed/`、`rebuild_train_table_fixed.py`）と旧表を、同じ設定で比べる。

## 比べるもの
- 特徴量セット: tp-03 で採用した 175 列（B0′ + 履歴 127 列）。前段モデルの 3 列（tp-04 の OOF）は旧表の特徴量から作られており、修正後の表では未再計算のため**入れない**。
- 旧表 vs 修正後の表、LightGBM（調整後パラメータ）、valid 2022〜2024 × 2 seed。同じ行どうしの勝率・連対・複勝 LogLoss と、レース単位の paired bootstrap の差（修正後 − 旧）。
- `track_bias_*` の列の変化量（変わった行の割合、欠損率の差）。

読み方: 差が 0 に近ければ「これまでの相対比較の結論は影響が小さい」と言える。修正後が悪くなる（LogLoss が増える）なら、旧表の値は先読みで楽観だったということ。test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ablation as ab, zoo, zoo_runner, feature_bakeoff as fb
from src.pipeline.models.compute import detect_compute
from src.pipeline.models.time_split import yearly_expanding_folds

OLD = REPO_ROOT / "data/local/modeling/training_pipeline"
NEW = REPO_ROOT / "data/local/modeling/training_pipeline_fixed"
OUT = NEW / "impact"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP22_THREADS", detect_compute(reserve_cores=6).threads))
SEEDS = [0, 1]
mlflow.set_experiment("keiba-tp-leak-impact")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
adopted = json.loads((OLD / "ablation_nospeed/adopted_features.json").read_text(encoding="utf-8"))
BASE = list(adopted["b0_features"]) + list(adopted["features"])
best = json.loads((OLD / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])


def load(work):
    df = ab.load_table(work / "features/train_table.parquet")
    df["race_id"] = df["race_id"].astype(str); df["horse_id"] = df["horse_id"].astype(str)
    df = df.sort_values(["date", "race_id", "horse_number"], kind="stable").reset_index(drop=True)
    return df


dfo, dfn = load(OLD), load(NEW)
print("旧表", dfo.shape, "| 修正後", dfn.shape)
key = ["race_id", "horse_id"]
common = dfo[key].merge(dfn[key], on=key, how="inner")
print("共通の行", len(common), "| 旧のみ", len(dfo) - len(common), "| 新のみ", len(dfn) - len(common))
missing = [c for c in BASE if c not in dfn.columns]
print("修正後の表に無い列:", missing[:10])

## track_bias 系の変化

In [ ]:
TB = [c for c in BASE if (c.startswith("track_bias") or "x_track_bias" in c) and str(dfo[c].dtype) not in ("category", "object")]
m = dfo[key + TB].merge(dfn[key + TB], on=key, suffixes=("_old", "_new"))
rows = []
for c in TB:
    a, b = m[c + "_old"].astype("float64").to_numpy(), m[c + "_new"].astype("float64").to_numpy()
    both = np.isfinite(a) & np.isfinite(b)
    rows.append({"列": c, "変わった行の割合": float(np.mean(~np.isclose(a[both], b[both]))) if both.any() else np.nan, "欠損率(旧)": float(np.mean(~np.isfinite(a))), "欠損率(新)": float(np.mean(~np.isfinite(b))),
                 "平均絶対差": float(np.mean(np.abs(a[both] - b[both]))) if both.any() else np.nan})
TBT = pd.DataFrame(rows); display(TBT.round(4)); TBT.to_csv(OUT / "track_bias_change.csv", index=False)

## 同じ設定で学習・評価（LightGBM、2 seed × valid 2022〜2024）

In [ ]:
def run(df, name):
    meta = df[["race_id", "horse_id", "finish_position", "year", "date"]].copy()
    folds = list(yearly_expanding_folds(df))
    res, oof = zoo_runner.run_oof(name, "lgb_binary", df[BASE], meta, folds, SEEDS, THREADS, params=dict(LP))
    return res, oof, meta


t0 = time.time()
res_o, oof_o, meta_o = run(dfo, "lgb_leak_old")
print(f"旧表: win {res_o['logloss_win'].mean():.5f} top2 {res_o['logloss_top2'].mean():.5f} top3 {res_o['logloss_top3'].mean():.5f} ({time.time() - t0:.0f}s)", flush=True)
res_n, oof_n, meta_n = run(dfn, "lgb_leak_fixed")
print(f"修正後: win {res_n['logloss_win'].mean():.5f} top2 {res_n['logloss_top2'].mean():.5f} top3 {res_n['logloss_top3'].mean():.5f} ({time.time() - t0:.0f}s)", flush=True)
fin_map = meta_o.set_index(key)["finish_position"]
cmp_ = fb.compare_oof(oof_o, oof_n, fin_map, "修正後 − 旧")
print({k: (round(v, 5) if isinstance(v, float) else v) for k, v in cmp_.items()})
by_year = pd.concat([res_o.assign(table="旧"), res_n.assign(table="修正後")]); display(by_year[["table", "fold", "logloss_win", "logloss_top2", "logloss_top3"]].round(5))

In [ ]:
report = {"git_commit": GIT_COMMIT, "rows_old": len(dfo), "rows_new": len(dfn), "rows_common": len(common),
          "old": {k: float(res_o[k].mean()) for k in ("logloss_win", "logloss_top2", "logloss_top3")},
          "fixed": {k: float(res_n[k].mean()) for k in ("logloss_win", "logloss_top2", "logloss_top3")},
          "diff_fixed_minus_old": {k: cmp_[k] for k in ("Δwin", "win_hi", "Δtop2", "Δtop3")}, "track_bias_change": TBT.round(5).to_dict("records")}
(OUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-22-leak-impact", tags={"notebook": "tp-22", "git_commit": GIT_COMMIT}):
    mlflow.log_metrics({"win_old": report["old"]["logloss_win"], "win_fixed": report["fixed"]["logloss_win"], "delta_win": float(cmp_["Δwin"]), "delta_top3": float(cmp_["Δtop3"])})
    mlflow.log_artifact(str(OUT / "report.json"), "leak_impact"); mlflow.log_artifact(str(OUT / "track_bias_change.csv"), "leak_impact")